<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/Audio_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai pydub ffmpeg-python

from openai import OpenAI
from google.colab import userdata
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
from pydub import AudioSegment

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# RECORD FROM MIC
def record_audio(seconds=10, filename="input.webm"):
    print(f"Recording for {seconds} seconds... Allow mic access")

    js = Javascript(f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        let chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(r => setTimeout(r, {seconds * 1000}));

        recorder.stop();
        await new Promise(r => recorder.onstop = r);

        const blob = new Blob(chunks, {{ type: 'audio/webm' }});
        const reader = new FileReader();

        return await new Promise(resolve => {{
            reader.onloadend = () => resolve(reader.result);
            reader.readAsDataURL(blob);
        }});
    }}
    recordAudio();
    """)

    display(js)
    audio_data = eval_js("recordAudio()")

    audio_bytes = b64decode(audio_data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    return filename

def convert_to_wav(input_file, output_file="audio.wav"):
    audio = AudioSegment.from_file(input_file)
    audio.export(output_file, format="wav")
    return output_file

def speech_to_text(audio_file):
    with open(audio_file, "rb") as f:
        response = client.audio.transcriptions.create(
            model="gpt-4o-transcribe",
            file=f
        )
    return response.text

seconds = int(input("How many seconds to record? (10-15 recommended): "))

raw_audio = record_audio(seconds)
wav_audio = convert_to_wav(raw_audio)

text = speech_to_text(wav_audio)

print("\n🎤 Transcribed Text:\n")
print(text)